# Evaluation

Loads a checkpoint - no training, so any run can be evaluated at any time through identical code.

Compare to DeepMoon's *post-CNN* 57% recall, not their 92% - that comes from merging ~120 views per crater, which this does not do.

In [ ]:
import sys
sys.path.append('../1_data_extraction')

import os
import numpy as np
import pandas as pd
import mlflow
import keras
import matplotlib.pyplot as plt

from crater_extraction import template_match_t, match_coords, filter_to_detectable, filter_edge_craters, truth_coords_for_patch
from LRO_data_class import getSplitIndices, percentileNormalise

## Run to evaluate

In [ ]:
# only cell that changes between runs - the rest is baked into the checkpoint

params = {
    'dataset': 'alltiles',                # 'single' | 'alltiles'
    'channels': 'both',                 # must match the trained run
    'n_filters': 32,
    'model': 'U-Net-v1',
    'seed': 42,                         # same sample across every run
    'n_sweep': 200,                     # val patches for the sweep
    'n_eval': 160,                      # test patches for the metrics
}

if params['dataset'] == 'single':
    PATCHES_DIR = '../3_pre_processing/lunar_patches'
    LABELS_CSV = '../2_data_preparation/filtered_labels.csv'
else:
    PATCHES_DIR = '../3_pre_processing/lunar_patches_alltiles'
    LABELS_CSV = '../2_data_preparation/filtered_labels_alltiles.csv'

CHECKPOINT_DIR = '../4_training/checkpoints'

run_name = f"{params['model']}_{params['channels']}_{params['n_filters']}f_s{params['seed']}"

# training writes one file per run, holding the best weights
CHECKPOINT = os.path.join(CHECKPOINT_DIR, f'{run_name}.keras')
HISTORY_CSV = os.path.join(CHECKPOINT_DIR, f'history_{run_name}.csv')

kept_labels = pd.read_csv(os.path.join(PATCHES_DIR, 'kept_labels.csv'), low_memory=False)
filtered_labels = pd.read_csv(LABELS_CSV)

train_idx, val_idx, test_idx = getSplitIndices(PATCHES_DIR)

print(f'train: {len(train_idx)}  val: {len(val_idx)}  test: {len(test_idx)}')
print(CHECKPOINT)

In [ ]:
model = keras.models.load_model(CHECKPOINT)

mlflow.set_tracking_uri('../4_training/mlruns')
mlflow.set_experiment('lunar-crater-detection')

with mlflow.start_run(run_name=f'eval-{run_name}') as run:
    mlflow.log_params(params)
    mlflow.log_param('checkpoint', CHECKPOINT)

    run_id = run.info.run_id

print(model.count_params(), 'params')

In [ ]:
# wac_col/wac_row survive only in kept_labels, so the lat/lon -> pixel map is fitted from it
# per tile: each tile counts pixels from its own corner
if 'tile' not in kept_labels.columns:
    kept_labels['tile'] = 'single'

tile_craters = {}

for tile_name in kept_labels['tile'].dropna().unique():

    tile_rows = kept_labels[kept_labels['tile'] == tile_name]
    crater_rows = tile_rows.dropna(subset=['LON_CIRC_IMG', 'wac_col'])

    col_fit = np.polyfit(crater_rows['LON_CIRC_IMG'], crater_rows['wac_col'], 1)
    row_fit = np.polyfit(crater_rows['LAT_CIRC_IMG'], crater_rows['wac_row'], 1)

    tile_wac_col = np.polyval(col_fit, filtered_labels['LON_CIRC_IMG'].values)
    tile_wac_row = np.polyval(row_fit, filtered_labels['LAT_CIRC_IMG'].values)

    tile_craters[tile_name] = (tile_wac_col, tile_wac_row, filtered_labels['DIAM_CIRC_IMG'].values)

    print(f'{tile_name}: lon -> col {col_fit[0]:.2f} px/deg     lat -> row {row_fit[0]:.2f} px/deg')

In [ ]:
# 1000 patches per file - held between calls. keyed on the dir too, or switching
# dataset would serve stale patches
loaded = {}


def patchInput(patch_idx):

    file_num = int(patch_idx // 1000)
    position = patch_idx % 1000

    if loaded.get('file') != (PATCHES_DIR, file_num):
        loaded['wac'] = np.load(os.path.join(PATCHES_DIR, f'X_wac_{file_num}.npz'))['arr_0']
        loaded['dem'] = np.load(os.path.join(PATCHES_DIR, f'X_dem_{file_num}.npz'))['arr_0']
        loaded['mask'] = np.load(os.path.join(PATCHES_DIR, f'X_mask_{file_num}.npz'))['arr_0']
        loaded['file'] = (PATCHES_DIR, file_num)

    wac_patch = percentileNormalise(loaded['wac'][position])
    dem_patch = percentileNormalise(loaded['dem'][position])

    if params['channels'] == 'both':
        return np.stack([wac_patch, dem_patch], axis=-1)

    if params['channels'] == 'wac':
        return wac_patch[..., None]

    return dem_patch[..., None]


def patchTruth(patch_idx):

    row = kept_labels.iloc[patch_idx]
    tile_wac_col, tile_wac_row, tile_diameters = tile_craters[row['tile']]

    truth = truth_coords_for_patch(row['center_col'], row['center_row'], row['patch_lat'],
                                   tile_wac_col, tile_wac_row, tile_diameters)

    return filter_edge_craters(filter_to_detectable(truth))

def patchMask(patch_idx):

    patchInput(patch_idx)

    return loaded['mask'][patch_idx % 1000]

## Threshold sweep

In [ ]:
# DeepMoon's 0.1 assumes unweighted BCE. focal sits lower - notes 17.3
# validation only, tuning on test contaminates everything after it

thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7]

sweep_rng = np.random.default_rng(params['seed'])
sweep_idx = np.sort(sweep_rng.choice(val_idx, size=params['n_sweep'], replace=False))

# same at every threshold, so built once
sweep_pred = []
sweep_truth = []
sweep_mask = []

for patch_idx in sweep_idx:

    sweep_pred.append(model.predict(patchInput(patch_idx)[None, ...], verbose=0)[0, :, :, 0])
    sweep_truth.append(patchTruth(patch_idx))
    sweep_mask.append(patchMask(patch_idx) > 0)

sweep_precision = []
sweep_recall = []
sweep_f1 = []

pixel_sweep_precision = []
pixel_sweep_recall = []

for threshold in thresholds:

    swept_match = 0
    swept_detected = 0
    swept_truth = 0

    swept_pixel_tp = 0
    swept_pixel_fp = 0
    swept_pixel_fn = 0

    for prediction, truth, true_rim in zip(sweep_pred, sweep_truth, sweep_mask):

        predicted_rim = prediction >= threshold

        both = (predicted_rim & true_rim).sum()

        swept_pixel_tp += both
        swept_pixel_fp += predicted_rim.sum() - both
        swept_pixel_fn += true_rim.sum() - both

        detections = filter_edge_craters(template_match_t(prediction.copy(), target_thresh=threshold))

        match_count, detection_count, truth_count, _, _, _ = match_coords(truth, detections)

        swept_match += match_count
        swept_detected += detection_count
        swept_truth += truth_count

    if swept_detected > 0:
        sweep_precision.append(swept_match / swept_detected)
    else:
        sweep_precision.append(0)

    if swept_truth > 0:
        sweep_recall.append(swept_match / swept_truth)
    else:
        sweep_recall.append(0)

    if sweep_precision[-1] + sweep_recall[-1] > 0:
        sweep_f1.append(2 * sweep_precision[-1] * sweep_recall[-1] / (sweep_precision[-1] + sweep_recall[-1]))
    else:
        sweep_f1.append(0)

    if swept_pixel_tp + swept_pixel_fp > 0:
        pixel_sweep_precision.append(swept_pixel_tp / (swept_pixel_tp + swept_pixel_fp))
    else:
        pixel_sweep_precision.append(0)

    if swept_pixel_tp + swept_pixel_fn > 0:
        pixel_sweep_recall.append(swept_pixel_tp / (swept_pixel_tp + swept_pixel_fn))
    else:
        pixel_sweep_recall.append(0)

    print(f'threshold {threshold}: P {sweep_precision[-1]:.3f}  R {sweep_recall[-1]:.3f}  F1 {sweep_f1[-1]:.3f}   | pixel P {pixel_sweep_precision[-1]:.3f}  R {pixel_sweep_recall[-1]:.3f}')

# optimum is interior - an all-zero column means widen the grid
best_threshold = thresholds[int(np.argmax(sweep_f1))]

print(f'best threshold: {best_threshold}   F1 {max(sweep_f1):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(thresholds, sweep_precision, marker='o', label='precision')
axes[0].plot(thresholds, sweep_recall, marker='o', label='recall')
axes[0].plot(thresholds, sweep_f1, marker='o', label='F1')
axes[0].set_xlabel('target_thresh')
axes[0].legend()

# the gap between the curves is what extraction costs
axes[1].plot(sweep_recall, sweep_precision, marker='o', label='crater')
axes[1].plot(pixel_sweep_recall, pixel_sweep_precision, marker='o', label='pixel')
axes[1].set_xlabel('recall')
axes[1].set_ylabel('precision')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.show()

## Crater-level metrics

In [ ]:
# test_idx directly, not the generator - it never says which patch it gave back

rng = np.random.default_rng(params['seed'])
# sorted so patches from one npz are consecutive
eval_idx = np.sort(rng.choice(test_idx, size=params['n_eval'], replace=False))

total_match = 0
total_detected = 0
total_truth = 0
total_multi_match = 0

all_matched_pairs = []
all_false_positives = []
all_truth_radii = []

pixel_tp = 0
pixel_fp = 0
pixel_fn = 0
pixel_tn = 0

for patch_idx in eval_idx:

    prediction = model.predict(patchInput(patch_idx)[None, ...], verbose=0)

    detections = filter_edge_craters(template_match_t(prediction[0, :, :, 0].copy(), target_thresh=best_threshold))

    truth = patchTruth(patch_idx)

    predicted_rim = prediction[0, :, :, 0] >= best_threshold
    true_rim = patchMask(patch_idx) > 0

    both = (predicted_rim & true_rim).sum()

    pixel_tp += both
    pixel_fp += predicted_rim.sum() - both
    pixel_fn += true_rim.sum() - both
    pixel_tn += predicted_rim.size - predicted_rim.sum() - true_rim.sum() + both

    match_count, detection_count, truth_count, matched_pairs, false_positives, multi_match_count = match_coords(truth, detections)

    total_match += match_count
    total_detected += detection_count
    total_truth += truth_count
    total_multi_match += multi_match_count

    if len(matched_pairs) > 0:
        all_matched_pairs.append(matched_pairs)

    if len(false_positives) > 0:
        all_false_positives.append(false_positives)

    if len(truth) > 0:
        all_truth_radii.append(truth[:, 2])

if len(all_matched_pairs) > 0:
    all_matched_pairs = np.vstack(all_matched_pairs)
else:
    all_matched_pairs = np.empty((0, 6))

if len(all_false_positives) > 0:
    all_false_positives = np.vstack(all_false_positives)
else:
    all_false_positives = np.empty((0, 3))

if len(all_truth_radii) > 0:
    all_truth_radii = np.concatenate(all_truth_radii)
else:
    all_truth_radii = np.empty(0)

print(f'TP: {total_match}   detected: {total_detected}   truth: {total_truth}')
print(f'detections claiming >1 truth crater: {total_multi_match}')

In [ ]:
# Crater-level metrics - the headline, as DeepMoon reports (2.6)
if total_detected > 0:
    precision = total_match / total_detected
else:
    precision = 0

if total_truth > 0:
    recall = total_match / total_truth
else:
    recall = 0

if precision + recall > 0:
    f1 = 2 * precision * recall / (precision + recall)
else:
    f1 = 0

# Pixel-level - a diagnostic, not the headline. separates "the model missed it"
# from "extraction lost it". accuracy is useless at 37:1 (all-zeros scores ~97%)
# so only the rim class. Zhang et al. 2024 report mIoU 75.2% for comparison
if pixel_tp + pixel_fp > 0:
    pixel_precision = pixel_tp / (pixel_tp + pixel_fp)
else:
    pixel_precision = 0

if pixel_tp + pixel_fn > 0:
    pixel_recall = pixel_tp / (pixel_tp + pixel_fn)
else:
    pixel_recall = 0

if pixel_precision + pixel_recall > 0:
    dice = 2 * pixel_precision * pixel_recall / (pixel_precision + pixel_recall)
else:
    dice = 0

if pixel_tp + pixel_fp + pixel_fn > 0:
    iou = pixel_tp / (pixel_tp + pixel_fp + pixel_fn)
else:
    iou = 0

print(f'crater  P {precision:.3f}   R {recall:.3f}   F1 {f1:.3f}')
print(f'pixel   P {pixel_precision:.3f}   R {pixel_recall:.3f}   Dice {dice:.3f}   IoU {iou:.3f}')

# logged to the training run so the nine runs can be compared in one place
with mlflow.start_run(run_id=run_id):
    mlflow.log_metric('precision', precision)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('f1', f1)
    mlflow.log_metric('multi_match', total_multi_match)
    mlflow.log_metric('target_thresh', best_threshold)
    mlflow.log_metric('pixel_precision', pixel_precision)
    mlflow.log_metric('pixel_recall', pixel_recall)
    mlflow.log_metric('pixel_dice', dice)
    mlflow.log_metric('pixel_iou', iou)

In [ ]:
# Results summary - the tables and figure for the report

summary = pd.DataFrame([
    {'level': 'crater', 'P': precision, 'R': recall, 'F1': f1,
     'TP': int(total_match), 'detected': int(total_detected), 'truth': int(total_truth)},
    {'level': 'pixel', 'P': pixel_precision, 'R': pixel_recall, 'F1': dice,
     'TP': int(pixel_tp), 'detected': int(pixel_tp + pixel_fp), 'truth': int(pixel_tp + pixel_fn)},
])

display(summary.round(3))

# crater level against the literature. DeepMoon post-CNN is the comparable column,
# not their post-processed 92% - that merges ~120 views per crater
this_run = f"this run - {params['channels']}, {params['dataset']}"

comparison = pd.DataFrame([
    {'source': 'DeepMoon post-CNN (Silburt 2019)', 'P': np.nan, 'R': 0.570, 'F1': np.nan},
    {'source': 'Silburt re-evaluated (Zhang 2024, T2)', 'P': 0.619, 'R': 0.873, 'F1': 0.725},
    {'source': 'Zhang et al. 2024', 'P': 0.783, 'R': 0.737, 'F1': 0.760},
    {'source': this_run, 'P': precision, 'R': recall, 'F1': f1},
])

display(comparison.round(3))

# row-normalised - at 37:1 the raw matrix is ~97% true negatives and says nothing
confusion = np.array([[pixel_tn, pixel_fp],
                      [pixel_fn, pixel_tp]], dtype=float)

normalised = confusion / confusion.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(4.5, 4))

ax.imshow(normalised, cmap='Blues', vmin=0, vmax=1)

for i in range(2):

    for j in range(2):

        if normalised[i, j] > 0.5:
            colour = 'white'
        else:
            colour = 'black'

        ax.text(j, i, f'{normalised[i, j]:.3f}\n{int(confusion[i, j]):,}',
                ha='center', va='center', color=colour)

ax.set_xticks([0, 1], ['pred background', 'pred rim'])
ax.set_yticks([0, 1], ['true background', 'true rim'])
ax.set_title('Pixel confusion, row-normalised')

plt.tight_layout()
plt.show()

## Visualisation

In [ ]:
# Loss curves
# from the CSV, not history - survives a crash, works on a loaded checkpoint

curves = pd.read_csv(HISTORY_CSV)

best_epoch = int(curves['val_loss'].idxmin())

plt.plot(curves['loss'], label='train')
plt.plot(curves['val_loss'], label='val')
plt.axvline(best_epoch, color='grey', linestyle='--', label=f'best epoch ({best_epoch + 1})')

plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()

In [ ]:
# Recall by crater diameter
# diameter_km = 2 * r_px * 0.1 -> 1-2 / 2-5 / 5-10 km are r = 5 / 10 / 25 / 50
# DeepMoon: recall drops above r = 15 px, 3 km here

bin_edges = [5, 10, 25, 50]
bin_labels = ['1-2 km', '2-5 km', '5-10 km']

matched_truth_radii = all_matched_pairs[:, 5]
matched_det_radii = all_matched_pairs[:, 2]
fp_radii = all_false_positives[:, 2]

bin_recall = []
bin_precision = []

for lower, upper in zip(bin_edges[:-1], bin_edges[1:]):

    truth_in_bin = ((all_truth_radii >= lower) & (all_truth_radii < upper)).sum()
    matched_in_bin = ((matched_truth_radii >= lower) & (matched_truth_radii < upper)).sum()

    if truth_in_bin > 0:
        bin_recall.append(matched_in_bin / truth_in_bin)
    else:
        bin_recall.append(0)

    # precision bins by DETECTED radius, recall by truth radius
    det_in_bin = ((matched_det_radii >= lower) & (matched_det_radii < upper)).sum()
    fp_in_bin = ((fp_radii >= lower) & (fp_radii < upper)).sum()

    if det_in_bin + fp_in_bin > 0:
        bin_precision.append(det_in_bin / (det_in_bin + fp_in_bin))
    else:
        bin_precision.append(0)

    print(f'{lower}-{upper} px:  R {matched_in_bin}/{truth_in_bin}   P {det_in_bin}/{det_in_bin + fp_in_bin}')

x = np.arange(len(bin_labels))

plt.bar(x - 0.2, bin_recall, 0.4, label='recall')
plt.bar(x + 0.2, bin_precision, 0.4, label='precision')

plt.xticks(x, bin_labels)
plt.ylabel('score')
plt.ylim(0, 1)
plt.title('Precision and recall by crater diameter')
plt.legend()
plt.show()

In [ ]:
# Crater size-frequency distribution
# parallel to the catalogue -> extra detections behave like real craters

all_detected_radii = np.concatenate([all_matched_pairs[:, 2], all_false_positives[:, 2]])

detected_diameters = all_detected_radii * 2 * 0.1
truth_diameters = all_truth_radii * 2 * 0.1

diameter_bins = np.logspace(np.log10(1), np.log10(10), 15)

plt.hist(truth_diameters, bins=diameter_bins, histtype='step', label='Robbins (in patch)')
plt.hist(detected_diameters, bins=diameter_bins, histtype='step', label='detected')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('diameter (km)')
plt.ylabel('count')
plt.legend()
plt.show()

In [ ]:
# Positional and radius error
# fractional, over the mean radius. DeepMoon medians <= 11% (Table 3.1)

mean_radius = (all_matched_pairs[:, 2] + all_matched_pairs[:, 5]) / 2

error_x = abs(all_matched_pairs[:, 0] - all_matched_pairs[:, 3]) / mean_radius
error_y = abs(all_matched_pairs[:, 1] - all_matched_pairs[:, 4]) / mean_radius
error_radius = abs(all_matched_pairs[:, 2] - all_matched_pairs[:, 5]) / mean_radius

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, values, name in zip(axes, [error_x, error_y, error_radius], ['x', 'y', 'radius']):

    ax.hist(values, bins=40)
    ax.axvline(np.median(values), color='red', linestyle='--')
    ax.set_title(f'{name}: median {np.median(values):.3f}')
    ax.set_xlabel('fractional error')

plt.show()

In [ ]:
# False positives
# Robbins incomplete near 1 km - some are real
# big ones on obvious craters are the >= 10 km label cut, not model error

n_show = 12

n_show = min(n_show, len(all_false_positives))
sample = all_false_positives[rng.choice(len(all_false_positives), n_show, replace=False)]

print('false positives: ', len(all_false_positives))
print(sample)

## Alignment check

WAC, stored mask and Robbins truth on one image - all derived independently. Rings on visible craters in both panels means the coordinate chain is sound.

In [ ]:
patch_idx = test_idx[0]

file_num = int(patch_idx // 1000)
position = patch_idx % 1000

raw_wac = np.load(os.path.join(PATCHES_DIR, f'X_wac_{file_num}.npz'))['arr_0']
raw_mask = np.load(os.path.join(PATCHES_DIR, f'X_mask_{file_num}.npz'))['arr_0']

wac_patch = percentileNormalise(raw_wac[position])
mask_patch = raw_mask[position]

truth = patchTruth(patch_idx)

fig, ax = plt.subplots(1, 2, figsize=(13, 6))

ax[0].imshow(wac_patch, cmap='gray')
ax[0].imshow(np.ma.masked_where(mask_patch == 0, mask_patch), cmap='autumn')
ax[0].set_title('WAC + stored mask')

ax[1].imshow(wac_patch, cmap='gray')

for x, yy, r in truth:
    ax[1].add_patch(plt.Circle((x, yy), r, fill=False, color='red'))

ax[1].set_title(f'WAC + Robbins ({len(truth)})')

plt.show()